In [ ]:
from functools import partial

import jax
from jax import numpy as jnp
from jax.sharding import PartitionSpec as P, NamedSharding, AxisType
import optax
import flax
from flax import nnx
# Ignore this if you are already running on a TPU or GPU
if not jax._src.xla_bridge.backends_are_initialized():
  jax.config.update('jax_num_cpu_devices', 8)

In [ ]:
# Create an auto-mode mesh of two dimensions and annotate each axis with a name.
rngs = nnx.Rngs(0)
auto_mesh = jax.make_mesh((2, 4), ('data', 'model'))
print(jax.devices())

## MLP sharding via `sharding=`

The MLP in `probjax.nn.nets.simple` accepts a `sharding` mesh and uses default partition rules for parameters and activations.


In [ ]:
from probjax.nn.nets.simple import MLP

with jax.set_mesh(auto_mesh):
  mlp = MLP(
      feature_dims=[1024, 4096, 1024],
      activation=jax.nn.gelu,
      sharding=auto_mesh,
      rngs=nnx.Rngs(0),
  )

  x = jax.device_put(rngs.normal((8, 1024)), P('data', None))
  y = mlp(x)
  print(y.shape, y.sharding.spec)
  jax.debug.visualize_array_sharding(y)  # already sharded!

## Transformer sharding via `sharding=`

Transformer blocks accept `sharding` and pass it through to internal layers.


In [ ]:
from probjax.nn.nets.transformer import Transformer

with jax.set_mesh(auto_mesh):
  transformer = Transformer(
      model_dim=128,
      num_heads=4,
      num_layers=2,
      attn_size=32,
      sharding=auto_mesh,
      rngs=nnx.Rngs(1),
  )

  x = jax.device_put(rngs.normal((8, 16, 128)), P('data', None, None))
  y = transformer(x)
  print(y.shape, y.sharding.spec)

In [ ]:
class DotReluDot(nnx.Module):
  def __init__(self, depth: int, rngs: nnx.Rngs):
    init_fn = nnx.initializers.lecun_normal()
    self.dot1 = nnx.Linear(
      depth, depth,
      kernel_init=nnx.with_partitioning(init_fn, (None, 'model')),
      use_bias=False,  # or use `bias_init` to give it annotation too
      rngs=rngs)
    self.w2 = nnx.Param(
      init_fn(rngs.params(), (depth, depth)),  # RNG key and shape for W2 creation
      sharding=('model', None),
    )

  def __call__(self, x: jax.Array):
    y = self.dot1(x)
    y = jax.nn.relu(y)
    y = jax.lax.with_sharding_constraint(y, P('data', 'model'))
    z = jnp.dot(y, self.w2[...])
    return z

In [ ]:
rngs = nnx.Rngs(0)
@jax.jit
def train_step(model, optimizer, x, y):
  def loss_fn(model: DotReluDot):
    y_pred = model(x)
    return jnp.mean((y_pred - y) ** 2)

  loss, grads = jax.value_and_grad(loss_fn)(model)
  optimizer.update(model, grads)
  return model, loss


with jax.set_mesh(auto_mesh):
  # Training data
  input = jax.device_put(rngs.normal((8, 1024)), P('data', None))
  label = jax.device_put(rngs.normal((8, 1024)), P('data', None))
  # Model and optimizer
  model = DotReluDot(1024, rngs=nnx.Rngs(0))
  optimizer = nnx.Optimizer(model, optax.adam(1e-3), wrt=nnx.Param)

  # The loop
  for i in range(5):
    model, loss = train_step(model, optimizer, input, label)
    print(loss)    # Model (over-)fitting to the labels quickly.

In [ ]:
print(input.shape)
print(input.sharding.spec)